# 3-Arms Technical Pipeline

Arms: Baseline (Title+Price), Volume (Title+Price+Volume), Full (All data including technical)

In [1]:
import os
import re
import asyncio
from datetime import datetime, timezone
import pandas as pd
from openai import AsyncOpenAI

# Config
DATA_PATH = "data/markets_microstructure_v2_v3_merged.csv"
OUT_DIR = "data/output"
OUT_PATH = os.path.join(OUT_DIR, "predictions_3arms_technical.csv")
MODEL = "llama-3.1-8b-instant"
CONCURRENCY = 8
TEMPERATURE = 0

# Create output directory
os.makedirs(OUT_DIR, exist_ok=True)

# Get API key - try environment variable, then .env file, then prompt
api_key = os.environ.get("GROQ_API_KEY")
if not api_key:
    try:
        from dotenv import load_dotenv
        load_dotenv()
        api_key = os.environ.get("GROQ_API_KEY")
    except:
        pass

if not api_key:
    api_key = input("Enter GROQ_API_KEY: ").strip()

client = AsyncOpenAI(
    base_url="https://api.groq.com/openai/v1",
    api_key=api_key,
)

print(f"Data: {DATA_PATH}")
print(f"Output: {OUT_PATH}")
print(f"Model: {MODEL}, Concurrency: {CONCURRENCY}")

Data: data/markets_microstructure_v2_v3_merged.csv
Output: data/output\predictions_3arms_technical.csv
Model: llama-3.1-8b-instant, Concurrency: 8


In [2]:
# Load data
df = pd.read_csv(DATA_PATH)
print(f"Loaded {len(df)} markets")
print(f"Columns: {len(df.columns)}")
df.head(2)

Loaded 1844 markets
Columns: 26


,category,event_ticker,event_title,market_ticker,market_title,yes_bid,yes_ask,mid_yes,last_price,spread_yes,...,time_to_close_hours,return_1h,return_6h,return_24h,trend_slope_24h,max_drawdown_24h,volatility_24h,vol_regime_shift,high_low_range_24h,candle_count
0,Climate and Weather,KXHMONTH-26JAN,This Jan 2026 is the hottest January ever?,KXHMONTH-26JAN,Hottest ever,0.00,0.02,0.020,0.02,NaN,...,143.74,-0.666667,-0.666667,-0.666667,-0.045172,0.750000,0.807498,1.1035,1.269231,22
1,Climate and Weather,KXHMONTHRANGE-26JAN,Jan 2026 temperature increase?,KXHMONTHRANGE-26JAN-T1.30,1.30001 or above,0.02,0.03,0.025,0.06,0.01,...,143.74,0.000000,0.000000,-0.666667,-0.234266,0.666667,0.392187,1.1443,1.043478,12


In [3]:
# Prompt templates
SYSTEM_PROMPT = """You are forecasting the probability this market resolves YES.
Treat mid_yes as a prior probability, then update/reconsider it using your reasoning of the additional signals provided for this branch to bias your output.
Output only one decimal number between 0 and 1 (example: 0.023). No words, no JSON, no punctuation."""

def build_baseline(row):
    return f"""You are forecasting the probability this market resolves YES.

Question: {row['event_title']}
mid_yes (prior): {row['mid_yes']:.2f}

Available signals: None (baseline - only question and prior).
Use your reasoning about the question to update the prior.
Output only the final decimal probability."""

def build_volume(row):
    return f"""You are forecasting the probability this market resolves YES.

Question: {row['event_title']}
mid_yes (prior): {row['mid_yes']:.2f}

Available signals: volume, volume_24h.
- volume: ${row['volume']:,.0f}
- volume_24h: ${row['volume_24h']:,.0f}

Update mid_yes using these signals: higher activity suggests more informed prior (smaller change); lower activity allows larger reconsideration.
Output only the final decimal probability."""

def build_full_technical(row):
    spread_text = f"{row['spread_yes']:.2f}" if pd.notna(row['spread_yes']) else "N/A"
    tick_text = f"{row['tick_size']:.2f}" if pd.notna(row['tick_size']) else "N/A"
    
    prompt = f"""You are forecasting the probability this market resolves YES.

Question: {row['event_title']}
mid_yes (prior): {row['mid_yes']:.2f}

Available signals: volume, volume_24h, open_interest, liquidity, spread_yes, tick_size, time_to_close_hours.
- volume: ${row['volume']:,.0f}
- volume_24h: ${row['volume_24h']:,.0f}
- open_interest: ${row['open_interest']:,.0f}
- liquidity: ${row['liquidity']:.2f}
- yes_bid: {row['yes_bid']:.2f}
- yes_ask: {row['yes_ask']:.2f}
- spread_yes: {spread_text}
- last_price: {row['last_price']:.2f}
- tick_size: {tick_text}
- time_to_close_hours: {row['time_to_close_hours']:.1f}"""
    
    if pd.notna(row.get('return_24h')):
        prompt += f"""

Additional technical signals: return_1h, return_6h, return_24h, trend_slope_24h, max_drawdown_24h, volatility_24h, vol_regime_shift, high_low_range_24h.
- return_1h: {row['return_1h']:.3f}
- return_6h: {row['return_6h']:.3f}
- return_24h: {row['return_24h']:.3f}
- trend_slope_24h: {row['trend_slope_24h']:.3f}
- max_drawdown_24h: {row['max_drawdown_24h']:.3f}
- volatility_24h: {row['volatility_24h']:.3f}
- vol_regime_shift: {row['vol_regime_shift']:.3f}
- high_low_range_24h: {row['high_low_range_24h']:.3f}"""
    
    prompt += """

Update mid_yes using these: higher activity/tighter spread => trust prior more (smaller change); lower activity/wider spread/large ticks => allow larger reconsideration.
Output only the final decimal probability."""
    return prompt

ARMS = {
    'baseline': build_baseline,
    'volume': build_volume,
    'full_technical': build_full_technical,
}

print("Prompt templates loaded")
print("\nBaseline example:")
print(build_baseline(df.iloc[0]))

Prompt templates loaded

Baseline example:
You are forecasting the probability this market resolves YES.

Question: This Jan 2026 is the hottest January ever?
mid_yes (prior): 0.02

Available signals: None (baseline - only question and prior).
Use your reasoning about the question to update the prior.
Output only the final decimal probability.


In [4]:
# API functions with retry logic
import random

def parse_probability(text):
    if not text or not text.strip():
        raise ValueError("Empty response")
    match = re.search(r'([01]?\.\d+|[01])', text.strip())
    if not match:
        raise ValueError(f"No probability in: '{text}'")
    p = float(match.group(1))
    if p > 1:
        p = p / 100
    if not (0 <= p <= 1):
        raise ValueError(f"Out of range: {p}")
    return p

async def query_market(row, arm_name, semaphore, max_retries=3):
    async with semaphore:
        prompt = ARMS[arm_name](row)
        raw = None
        p_yes = None
        error = None
        
        for attempt in range(1, max_retries + 1):
            try:
                response = await client.chat.completions.create(
                    model=MODEL,
                    messages=[
                        {"role": "system", "content": SYSTEM_PROMPT},
                        {"role": "user", "content": prompt}
                    ],
                    temperature=TEMPERATURE,
                )
                raw = response.choices[0].message.content.strip()
                p_yes = parse_probability(raw)
                error = None
                break
            except ValueError as e:
                # Parsing error: retrying usually won't help
                error = f"Parse: {str(e)}"
                break
            except Exception as e:
                error = f"API: {str(e)}"
                if attempt < max_retries:
                    # Exponential backoff + jitter
                    await asyncio.sleep((2 ** (attempt - 1)) + random.random())
                else:
                    break
        
        return {
            'timestamp': datetime.now(timezone.utc).isoformat(),
            'model': MODEL,
            'arm': arm_name,
            'event_ticker': row['event_ticker'],
            'market_ticker': row['market_ticker'],
            'title': row['event_title'],
            'raw_response': raw,
            'p_yes': p_yes,
            'error': error,
            'has_technical': pd.notna(row.get('return_24h')),
            'attempts': attempt
        }

print("API functions loaded with retry logic (max 3 attempts with exponential backoff)")

API functions loaded with retry logic (max 3 attempts with exponential backoff)


In [5]:
# Test with 5 markets
async def test_multiple():
    sem = asyncio.Semaphore(CONCURRENCY)
    test_markets = df.head(5)
    results = []
    
    print(f"Testing {len(test_markets)} markets x 3 arms = {len(test_markets) * 3} calls\n")
    
    for idx, (_, market) in enumerate(test_markets.iterrows(), 1):
        print(f"[{idx}/5] {market['event_ticker']}: {market['event_title'][:60]}...")
        
        for arm in ['baseline', 'volume', 'full_technical']:
            result = await query_market(market, arm, sem)
            results.append(result)
            status = "OK" if not result['error'] else f"ERROR: {result['error']}"
            p_str = f"{result['p_yes']:.3f}" if result['p_yes'] else "None"
            print(f"  {arm:15s}: {p_str} ({result['attempts']} attempt(s)) - {status}")
    
    df_test = pd.DataFrame(results)
    success = df_test['error'].isna().sum()
    total = len(df_test)
    
    print(f"\n{'='*70}")
    print(f"Test Results: {success}/{total} successful ({success/total*100:.1f}%)")
    print(f"{'='*70}")
    
    return df_test

df_test = await test_multiple()
df_test[['event_ticker', 'arm', 'p_yes', 'error', 'attempts', 'has_technical']]

Testing 5 markets x 3 arms = 15 calls

[1/5] KXHMONTH-26JAN: This Jan 2026 is the hottest January ever?...
  baseline       : 0.005 (1 attempt(s)) - OK
  volume         : 0.021 (1 attempt(s)) - OK
  full_technical : 0.021 (1 attempt(s)) - OK
[2/5] KXHMONTHRANGE-26JAN: Jan 2026 temperature increase?...
  baseline       : 0.035 (1 attempt(s)) - OK
  volume         : 0.042 (1 attempt(s)) - OK
  full_technical : 0.032 (1 attempt(s)) - OK
[3/5] KXHMONTHRANGE-26JAN: Jan 2026 temperature increase?...
  baseline       : 0.100 (1 attempt(s)) - OK
  volume         : 0.100 (1 attempt(s)) - OK
  full_technical : 0.100 (1 attempt(s)) - OK
[4/5] KXHMONTHRANGE-26JAN: Jan 2026 temperature increase?...
  baseline       : 0.500 (1 attempt(s)) - OK
  volume         : 0.500 (1 attempt(s)) - OK
  full_technical : 0.500 (1 attempt(s)) - OK
[5/5] KXHMONTHRANGE-26JAN: Jan 2026 temperature increase?...
  baseline       : 0.050 (1 attempt(s)) - OK
  volume         : 0.015 (1 attempt(s)) - OK
  full_technical : 

,event_ticker,arm,p_yes,error,attempts,has_technical
0,KXHMONTH-26JAN,baseline,0.005,None,1,True
1,KXHMONTH-26JAN,volume,0.021,None,1,True
2,KXHMONTH-26JAN,full_technical,0.021,None,1,True
3,KXHMONTHRANGE-26JAN,baseline,0.035,None,1,True
4,KXHMONTHRANGE-26JAN,volume,0.042,None,1,True
5,KXHMONTHRANGE-26JAN,full_technical,0.032,None,1,True
6,KXHMONTHRANGE-26JAN,baseline,0.100,None,1,True
7,KXHMONTHRANGE-26JAN,volume,0.100,None,1,True
8,KXHMONTHRANGE-26JAN,full_technical,0.100,None,1,True
9,KXHMONTHRANGE-26JAN,baseline,0.500,None,1,True


In [ ]:
# Full pipeline with progress tracking
async def run_pipeline(df_markets):
    sem = asyncio.Semaphore(CONCURRENCY)
    tasks = []
    completed = 0
    successful = 0
    total_calls = len(df_markets) * 3
    
    # Create all tasks
    for _, row in df_markets.iterrows():
        for arm_name in ARMS.keys():
            tasks.append(query_market(row, arm_name, sem))
    
    print(f"Starting {total_calls} API calls ({len(df_markets)} markets x 3 arms)")
    print(f"Started: {datetime.now().strftime('%H:%M:%S')}")
    print(f"Progress will update every 100 calls...")
    print()
    
    # Process tasks with progress tracking
    results = []
    for i, task in enumerate(asyncio.as_completed(tasks), 1):
        result = await task
        results.append(result)
        
        # Track success
        if result['error'] is None:
            successful += 1
        
        # Progress updates every 100 calls
        if i % 100 == 0 or i == total_calls:
            elapsed = (datetime.now().hour * 3600 + datetime.now().minute * 60 + datetime.now().second)
            rate = i / max(elapsed, 1) * 60  # calls per minute
            remaining = (total_calls - i) / max(rate, 1)
            
            print(f"[{i:4d}/{total_calls}] Completed: {successful}/{i} successful ({successful/i*100:.1f}%) | "
                  f"Rate: {rate:.1f} calls/min | ETA: ~{remaining:.1f} min")
    
    print(f"\nCompleted: {datetime.now().strftime('%H:%M:%S')}")
    print(f"Final: {successful}/{total_calls} successful ({successful/total_calls*100:.1f}%)")
    
    return pd.DataFrame(results)

df_results = await run_pipeline(df)

# Save
df_results.to_csv(OUT_PATH, index=False)
print(f"\nSaved {len(df_results)} predictions to {OUT_PATH}")

Starting 5532 API calls (1844 markets x 3 arms)
Started: 02:35:07
Progress will update every 100 calls...

[ 100/5532] Completed: 73/100 successful (73.0%) | Rate: 0.6 calls/min | ETA: ~5432.0 min
[ 200/5532] Completed: 134/200 successful (67.0%) | Rate: 1.3 calls/min | ETA: ~4227.8 min
[ 300/5532] Completed: 193/300 successful (64.3%) | Rate: 1.9 calls/min | ETA: ~2801.4 min
[ 400/5532] Completed: 251/400 successful (62.7%) | Rate: 2.5 calls/min | ETA: ~2086.8 min
[ 500/5532] Completed: 311/500 successful (62.2%) | Rate: 3.0 calls/min | ETA: ~1657.7 min
[ 600/5532] Completed: 371/600 successful (61.8%) | Rate: 3.6 calls/min | ETA: ~1370.7 min
[ 700/5532] Completed: 429/700 successful (61.3%) | Rate: 4.1 calls/min | ETA: ~1165.4 min
[ 800/5532] Completed: 490/800 successful (61.3%) | Rate: 4.7 calls/min | ETA: ~1010.6 min
[ 900/5532] Completed: 551/900 successful (61.2%) | Rate: 5.2 calls/min | ETA: ~889.9 min
[1000/5532] Completed: 609/1000 successful (60.9%) | Rate: 5.7 calls/min | E

In [ ]:
# Summary
total = len(df_results)
success = df_results['error'].isna().sum()
errors = df_results['error'].notna().sum()

print(f"Total: {total}")
print(f"Success: {success} ({success/total*100:.1f}%)")
print(f"Errors: {errors} ({errors/total*100:.1f}%)")

print("\nBy arm:")
for arm in ['baseline', 'volume', 'full_technical']:
    df_arm = df_results[df_results['arm'] == arm]
    arm_success = df_arm['error'].isna().sum()
    print(f"  {arm}: {arm_success}/{len(df_arm)} ({arm_success/len(df_arm)*100:.1f}%)")

df_full = df_results[df_results['arm'] == 'full_technical']
with_tech = df_full['has_technical'].sum()
print(f"\nTechnical data: {with_tech}/{len(df_full)} markets ({with_tech/len(df_full)*100:.1f}%)")

if errors > 0:
    print("\nTop errors:")
    print(df_results[df_results['error'].notna()]['error'].value_counts().head(3))

print("\nSample predictions:")
df_results[df_results['error'].isna()].head(6)[['event_ticker', 'arm', 'p_yes', 'has_technical']]